In [3]:
%pip install pandas numpy os scikit-learn

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler

In [5]:
folder_path = r'C:\AirPollutionPrediction-CNN-BiLSTM\data'
input_name = 'Air Quality Ho Chi Minh City.csv'
output_name = 'Air_Quality_Processed.csv'

input_path = os.path.join(folder_path, input_name)
output_path = os.path.join(folder_path, output_name)

In [6]:
if not os.path.exists(input_path):
    print(f"Không tìm thấy file")
else:
    df = pd.read_csv(input_path)
    # Chuyển đổi cột date sang định dạng thời gian
    df['date'] = pd.to_datetime(df['date'], dayfirst=True)
    # Sắp xếp để đảm bảo tính tuần tự của chuỗi thời gian
    df = df.sort_values(by=['Station_No', 'date'])
    print(f"Tổng số hàng ban đầu: {len(df)}")
    display(df.head())

,date,Station_No,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity
0,2021-02-23 21:00:00,1,32.935714,15.604762,55.431381,1330.451429,112.740762,393.000000,28.361905,63.188095
1,2021-02-23 22:00:00,1,30.932353,14.594118,58.197176,1200.603529,112.366471,377.588235,28.320588,63.773529
2,2021-02-23 23:00:00,1,27.645000,13.436667,55.029433,1177.897000,112.700433,372.476667,28.336667,64.205000
3,2021-02-24 00:00:00,1,24.380000,12.365000,54.767700,1267.476000,112.480867,389.070000,28.305000,64.735000
4,2021-02-24 01:00:00,1,22.521667,11.636667,53.786200,1322.293000,114.331500,393.000000,28.300000,65.188333


Tổng số hàng ban đầu: 52548


In [9]:
#Loại bỏ giá trị bất thường
df.loc[(df['Temperature'] > 60) | (df['Temperature'] < 10), 'Temperature'] = np.nan
df.loc[df['Humidity'] > 100, 'Humidity'] = np.nan

In [10]:
cols_sensor = ['TSP', 'PM2.5', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity']

# Đặt index là date để resample
if 'date' in df.columns:
    df = df.set_index('date')

df = df.groupby('Station_No')[cols_sensor].resample('h').mean().reset_index()
print(f"Tổng số hàng sau khi chuẩn hóa: {len(df)}")
display(df.head())

Tổng số hàng sau khi chuẩn hóa: 69515


,Station_No,date,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity
0,1,2021-02-23 21:00:00,32.935714,15.604762,55.431381,1330.451429,112.740762,393.000000,28.361905,63.188095
1,1,2021-02-23 22:00:00,30.932353,14.594118,58.197176,1200.603529,112.366471,377.588235,28.320588,63.773529
2,1,2021-02-23 23:00:00,27.645000,13.436667,55.029433,1177.897000,112.700433,372.476667,28.336667,64.205000
3,1,2021-02-24 00:00:00,24.380000,12.365000,54.767700,1267.476000,112.480867,389.070000,28.305000,64.735000
4,1,2021-02-24 01:00:00,22.521667,11.636667,53.786200,1322.293000,114.331500,393.000000,28.300000,65.188333


In [11]:
#Xử lý outlier
cols_sensor = ['TSP', 'PM2.5', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity']

def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return series.clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

for col in cols_sensor:
    df[col] = df.groupby('Station_No')[col].transform(cap_outliers)
    
    df[col] = df.groupby('Station_No')[col].transform(
        lambda x: x.interpolate(method='linear').ffill().bfill()
    )


In [12]:
# Đặc trưng thời gian
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek

# Đặc trưng tuần hoàn (Sin/Cos)
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

# Đặc trưng trễ (Lag) và Trung bình trượt (Rolling)
df['PM2.5_lag1'] = df.groupby('Station_No')['PM2.5'].shift(1)
df['PM2.5_roll3'] = df.groupby('Station_No')['PM2.5'].transform(lambda x: x.rolling(window=3).mean())

# Lấp đầy các giá trị NaN mới sinh ra do lag/rolling
df = df.ffill().bfill()

In [13]:
# Chọn các đặc trưng để đưa vào mô hình
final_features = cols_sensor + ['hour_sin', 'hour_cos', 'PM2.5_lag1', 'PM2.5_roll3']

scaler = MinMaxScaler()
df[final_features] = scaler.fit_transform(df[final_features])

In [14]:
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    
#In ra file
df.to_csv(output_path, index=False)
print(f"File đã được lưu tại: {output_path}")
display(df.head())

File đã được lưu tại: C:\AirPollutionPrediction-CNN-BiLSTM\data\Air_Quality_Processed.csv


,Station_No,date,TSP,PM2.5,O3,CO,NO2,SO2,Temperature,Humidity,hour,day_of_week,hour_sin,hour_cos,PM2.5_lag1,PM2.5_roll3
0,1,2021-02-23 21:00:00,0.231599,0.283775,0.241747,0.289115,0.244509,0.587964,0.418002,0.584469,21,1,0.146447,0.853553,0.283775,0.264506
1,1,2021-02-23 22:00:00,0.217511,0.265396,0.253809,0.289115,0.243697,0.564752,0.415874,0.591220,22,1,0.250000,0.933013,0.283775,0.264506
2,1,2021-02-23 23:00:00,0.194395,0.244347,0.239994,0.289115,0.244422,0.557054,0.416702,0.596195,23,1,0.370590,0.982963,0.265396,0.264506
3,1,2021-02-24 00:00:00,0.171436,0.224859,0.238853,0.289115,0.243946,0.582045,0.415072,0.602306,0,2,0.500000,1.000000,0.244347,0.244867
4,1,2021-02-24 01:00:00,0.158369,0.211614,0.234572,0.289115,0.247959,0.587964,0.414814,0.607533,1,2,0.629410,0.982963,0.224859,0.226940
